# Part B · NB0 — Data Prep & Split-Role Mapping  ·  `pavecrack1300-partb-data-prep`

**Course:** CSE 438 — Digital Image Processing · Dept. of CSE, **East West University** · Summer 2026
**Instructor:** _`<fill instructor>`_ · **Group:** _`<fill group>`_ · **Section:** _`<3 or 4>`_

| # | Name | Student ID |
|---|---|---|
| 1 | MD. Asif Hossain | 2022-3-60-007 |
| 2 | _`<member 2>`_ | _`<id>`_ |
| 3 | _`<member 3>`_ | _`<id>`_ |
| 4 | _`<member 4>`_ | _`<id>`_ |

**Dataset:** PaveCrack1300 (same as Part A) · binary crack / background.

---
### What this notebook does (Task A)
Part B reuses the **exact leakage-safe split from Part A** — we do **not** re-split or re-shuffle. We only give
each of the three existing splits a **new role**:

| Part-A split | New Part-B role | Used how |
|---|---|---|
| **train** (909) | **Unlabelled** SSL pretraining corpus | images only, masks discarded |
| **validation** (195) | **Labelled** fine-tuning set | image + mask (the small labelled data) |
| **test** (196) | **Monitoring + final evaluation** | image + mask, no augmentation |

**Label-efficiency ratio:** we fine-tune on **195** labelled images vs **909** the Part-A baseline used ≈ **21%**.

> ⚠️ **Test-split double role (report §6.3).** The test split is used *both* to select the best checkpoint during
> fine-tuning *and* to report final metrics. This is a deliberate simplification (the labelled val set is too small
> to also hold out a monitor slice); it mildly **inflates** reported numbers vs a truly-untouched test set. Stated
> here and in the report.

> **Resolution note.** Part A trained at 512². Part B standardizes on **224²** — the one size that works for every
> SSL encoder (ResNet, MAE ViT-B/16 → 14×14 tokens, DINOv2 ViT-S/14 → 16×16 tokens) and fits the Kaggle T4.

This notebook builds the three loaders, runs a sanity grid, and saves `partB_roles.json` for the method notebooks.

### Before you run
**+ Add Input:** the raw **PaveCrack1300** dataset **and** the committed **Part-A NB0 output** (which contains
`split.json`). CPU is fine for this notebook.

## 1 · Setup & config

In [ ]:
import os, glob, json, random, warnings
from pathlib import Path
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
warnings.filterwarnings("ignore")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=False)
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

print("torch", torch.__version__, "| albumentations", A.__version__)

# ============================ CONFIG ============================
SEED          = 42
IMG_SIZE      = 224                 # Part-B resolution (Part A used 512)
NUM_CLASSES   = 2
CLASS_NAMES   = ["background", "crack"]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
WORK          = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
# ===============================================================
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()
print("config loaded · IMG_SIZE", IMG_SIZE)

## 2 · Load the Part-A split and remap roles (no re-shuffle)
We locate Part-A's `split.json` with a recursive glob, then resolve absolute image/mask paths from the raw dataset
(the same discovery Part A used). We assign roles and report the exact inherited counts.

In [ ]:
def _find(pat):
    h = glob.glob(pat, recursive=True); return h[0] if h else None

split_path = _find("/kaggle/input/**/split.json")
assert split_path, "Part-A split.json not found — attach the committed Part-A NB0 output."
split = json.load(open(split_path))
print("loaded Part-A split from:", split_path)
print("Part-A counts:", split["counts"], "| seed:", split["seed"])

IMGS  = glob.glob("/kaggle/input/**/images/images/*.jpg", recursive=True) or \
        glob.glob("/kaggle/input/**/images/**/*.jpg", recursive=True)
MASKS = glob.glob("/kaggle/input/**/masks/**/*.png", recursive=True)
PATH_BY_NAME = {Path(p).name: p for p in IMGS}
MASK_BY_NAME = {Path(p).name: p for p in MASKS}
assert PATH_BY_NAME and MASK_BY_NAME, "raw PaveCrack1300 not found — attach the dataset."

roles = {"train": split["train"], "val": split["val"], "test": split["test"]}
n_tr, n_va, n_te = len(roles["train"]), len(roles["val"]), len(roles["test"])
print(f"\nPart-B role mapping:")
print(f"  UNLABELLED pretrain  <- Part-A train : {n_tr} images (masks discarded)")
print(f"  LABELLED  fine-tune  <- Part-A val   : {n_va} images (+ masks)")
print(f"  MONITOR + EVAL       <- Part-A test  : {n_te} images (+ masks)")
print(f"  label-efficiency ratio = {n_va}/{n_tr} = {100*n_va/n_tr:.1f}% of the Part-A labelled budget")

# save artifact for the method notebooks
partB = dict(resolution=IMG_SIZE, num_classes=NUM_CLASSES, class_names=CLASS_NAMES,
             roles=dict(unlabelled_pretrain="train", labelled_finetune="val", monitor_and_eval="test"),
             counts=dict(train=n_tr, val=n_va, test=n_te),
             label_efficiency_ratio=round(n_va/n_tr, 4),
             seed=SEED, note="test split serves double role (monitor + final eval)",
             train=roles["train"], val=roles["val"], test=roles["test"])
(WORK/"partB_roles.json").write_text(json.dumps(partB, indent=2))
print("\nsaved -> partB_roles.json")

## 3 · Augmentation pipelines (per role)
- **Unlabelled (SSL) loader** — yields **two augmented views** of the same image, **no mask**. This is the
  SimCLR-style view generator (each method may customize it in its own notebook). We keep flips + random-resized-crop
  + mild colour/blur, and **drop grayscale** (asphalt is already near-grayscale).
- **Labelled (fine-tune) loader** — the **Part-A synchronized image+mask** Albumentations pipeline, at 224².
- **Test loader** — **no augmentation** (resize + normalize only); used for both monitoring and final evaluation.

In [ ]:
# ---- SSL two-view transform (image only) ----
ssl_view_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), antialias=True),
    T.RandomHorizontalFlip(0.5),
    T.RandomVerticalFlip(0.5),                                   # top-down view -> vflip is valid
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
    T.RandomApply([T.GaussianBlur(23, (0.1, 2.0))], p=0.5),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---- labelled synchronized transform (image + mask), version-robust ----
def _try(cls, *variants):
    last=None
    for kw in variants:
        try: return cls(**kw)
        except (TypeError, ValueError) as e: last=e
    raise RuntimeError(f"{cls.__name__}: {last}")
def _geom(p=0.5):
    if hasattr(A, "ShiftScaleRotate"):
        try: return A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15,
                                       border_mode=cv2.BORDER_REFLECT_101, p=p)
        except (TypeError, ValueError): pass
    return _try(A.Affine, dict(translate_percent=0.0625, scale=(0.9,1.1), rotate=(-15,15),
                               border_mode=cv2.BORDER_REFLECT_101, p=p),
                dict(translate_percent=0.0625, scale=(0.9,1.1), rotate=(-15,15), p=p))

label_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(0.5), A.VerticalFlip(0.5), A.RandomRotate90(0.5), _geom(0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2(),
])
test_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])

class SSLTwoView(Dataset):
    def __init__(self, recs, tf): self.recs, self.tf = recs, tf
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        img = Image.open(PATH_BY_NAME[self.recs[i]["image"]]).convert("RGB")
        return self.tf(img), self.tf(img)

class LabeledSeg(Dataset):
    def __init__(self, recs, aug): self.recs, self.aug = recs, aug
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        img = np.array(Image.open(PATH_BY_NAME[r["image"]]).convert("RGB"))
        m   = (np.array(Image.open(MASK_BY_NAME[r["mask"]])) > 0).astype("uint8")
        o = self.aug(image=img, mask=m)
        return o["image"], o["mask"].long()

ssl_ds  = SSLTwoView(roles["train"], ssl_view_tf)     # unlabelled pretraining
label_ds = LabeledSeg(roles["val"],  label_tf)        # labelled fine-tune
test_ds  = LabeledSeg(roles["test"], test_tf)         # monitor + eval

v1, v2 = ssl_ds[0]
xb, yb = label_ds[0]
tb, tm = test_ds[0]
print("SSL two-view :", tuple(v1.shape), "+", tuple(v2.shape))
print("labelled     :", tuple(xb.shape), "| mask", tuple(yb.shape), "classes", torch.unique(yb).tolist())
print("test         :", tuple(tb.shape), "| mask", tuple(tm.shape))

## 4 · Task A sanity grid (required quality gate)
Two checks before any training: **(a)** two augmented views of the *same* train-split image (confirms the SSL
view generator produces diverse-but-related crops), and **(b)** a validation-split image with its mask overlay
(confirms the labelled image+mask pipeline stays aligned).

In [ ]:
def denorm(t):
    x = t.permute(1,2,0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(x, 0, 1)

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
# (a) two SSL views of the same image
a1, a2 = ssl_ds[0]
ax[0].imshow(denorm(a1)); ax[0].set_title("SSL view 1 (train img)", fontsize=9); ax[0].axis("off")
ax[1].imshow(denorm(a2)); ax[1].set_title("SSL view 2 (same img)", fontsize=9); ax[1].axis("off")
# (b) labelled val image + mask overlay
xb, yb = label_ds[0]
img = denorm(xb); m = yb.numpy()
ov = img.copy(); ov[m == 1] = 0.45*ov[m == 1] + 0.55*np.array([1, 0, 0])
ax[2].imshow(img);           ax[2].set_title("labelled val image", fontsize=9); ax[2].axis("off")
ax[3].imshow(ov);            ax[3].set_title("val image + mask overlay", fontsize=9); ax[3].axis("off")
plt.suptitle("Task A — sanity grid: SSL two-view (left) · labelled image+mask (right)", y=1.02)
plt.tight_layout(); plt.savefig(WORK/"partB_taskA_sanity.png", dpi=110, bbox_inches="tight"); plt.show()

## 5 · NB0 complete
**Saved to `/kaggle/working`:** `partB_roles.json` (role mapping + file lists + counts) and the sanity figure.

**Next:** **Save Version → Commit**. Each SSL method notebook (SimCLR, BYOL, MAE, DINOv2) will attach this output
plus the raw dataset, load `partB_roles.json`, and reuse this identical role mapping so all four are apples-to-apples.

**Compliance (Task A):** Part-A split loaded unmodified ✓ · roles mapped (train→unlabelled, val→labelled,
test→monitor+eval) ✓ · counts + label-efficiency ratio reported ✓ · SSL two-view + labelled + test loaders built ✓ ·
sanity grid shown ✓ · file lists saved as an artifact ✓.